# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShivanshRastogi315/FlyRank_Internship-repo/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of Analysis:** One row represents exactly one unique published content item (a webpage), identified by `content_id`.

**Time Window:** The primary performance window is a **90-day trailing snapshot** (represented by fields like `impressions_90d`, `sessions_90d`). We also have 30-day segmented snapshots (`last_30d`, `prev_30d`) to calculate momentum and trends, bounded by the `content_age_days`.

In [3]:
import pandas as pd
import numpy as np

# Load the local dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# 1. Verify Grain (One row = one unique content_id)
total_rows = len(df)
unique_content = df['content_id'].nunique()
is_unique = total_rows == unique_content

print(f"Total Rows: {total_rows}")
print(f"Unique Content IDs: {unique_content}")
print(f"Grain Verified (1 row = 1 page): {is_unique}")

Total Rows: 30000
Unique Content IDs: 30000
Grain Verified (1 row = 1 page): True


## 2. Fields: feature / label / context / excluded

*   **Features (Predictors):** `search_volume`, `word_count`, `char_count`, `content_age_days`, `days_since_last_update`, `avg_position`, `engagement_rate`, `scroll_rate`. (These describe the page's structure and general SERP footprint).
*   **Label (Target):** `ai_sessions_90d` and `ai_traffic_pct`. (What we are trying to isolate).
*   **Context (Metadata):** `content_id`, `client_id`, `content_type`, `main_intent`. (Used for grouping and holdout validation, strictly not for training).
*   **Excluded (Denylist):** 
    *   `trend_direction` / `trend_pct` (Post-facto target leakage).
    *   `impressions_90d` / `clicks_90d` / `sessions_90d` (Highly correlated with overall site authority rather than AI-archetype fit, which would skew the Isolation Forest).

In [4]:
features = ['search_volume', 'word_count', 'char_count', 'content_age_days', 
            'days_since_last_update', 'avg_position', 'engagement_rate', 'scroll_rate']
labels = ['ai_sessions_90d', 'ai_traffic_pct']
context = ['content_id', 'client_id', 'content_type', 'main_intent']
excluded = ['trend_direction', 'trend_pct', 'impressions_90d', 'clicks_90d', 'sessions_90d']

# Verify all categorized fields exist in the dataframe
all_declared_fields = features + labels + context + excluded
missing_fields = [f for f in all_declared_fields if f not in df.columns]

print(f"All declared fields exist in dataset: {len(missing_fields) == 0}")
if missing_fields:
    print(f"Missing fields: {missing_fields}")

All declared fields exist in dataset: True


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
print("--- DATA CONTRACT VERIFICATION ---")

# 1. Missing Values in Features
print("\nMissing values in our feature set:")
missing_vals = df[features].isnull().sum()
print(missing_vals[missing_vals > 0])
if missing_vals.sum() == 0:
    print("Zero missing values in primary features!")

# 2. Verify Time Windows (Logical bounds)
# Age cannot be negative, and days since update cannot exceed content age
invalid_age = df[df['content_age_days'] < 0].shape[0]
invalid_update = df[df['days_since_last_update'] > df['content_age_days']].shape[0]

print(f"\nRows with negative content age: {invalid_age}")
print(f"Rows where update is older than creation: {invalid_update}")

# 3. Label Sparsity (The Extreme Value Problem)
ai_positive_count = (df['ai_sessions_90d'] > 0).sum()
ai_positive_pct = (ai_positive_count / total_rows) * 100

print(f"\nAI-Traffic Sparsity:")
print(f"Pages with AI Traffic: {ai_positive_count} out of {total_rows} ({ai_positive_pct:.2f}%)")

--- DATA CONTRACT VERIFICATION ---

Missing values in our feature set:
search_volume    2468
word_count       7699
char_count       7699
scroll_rate       125
dtype: int64

Rows with negative content age: 0
Rows where update is older than creation: 0

AI-Traffic Sparsity:
Pages with AI Traffic: 1930 out of 30000 (6.43%)


## 4. Data limits

**What this data can NEVER tell us:**
1.  **Temporal Causality:** The 90-day metrics are aggregated snapshots. We cannot tell *on which specific day* a page gained AI traffic, meaning we cannot correlate traffic spikes with specific algorithm updates.
2.  **Extreme Class Imbalance:** Because only ~6.4% of pages have AI sessions, our dataset is heavily biased towards traditional search behaviors. We lack a massive volume of positive AI examples.
3.  **Blind Spots:** `search_volume` and `avg_position` are likely blended across all search engines (Google, Bing, etc.), meaning we cannot isolate Google's AI Overviews from Bing Chat.

In [6]:
# Prove the data limit mathematically: The overlapping domains problem
# Are there clients who have ZERO AI traffic across their entire property?

client_ai_summary = df.groupby('client_id')['ai_sessions_90d'].sum()
clients_with_zero_ai = (client_ai_summary == 0).sum()
total_clients = df['client_id'].nunique()

print("--- DATA LIMIT: CLIENT BIAS ---")
print(f"Total Unique Clients: {total_clients}")
print(f"Clients with exactly ZERO AI traffic across all pages: {clients_with_zero_ai}")
print(f"Percentage of blind-spot clients: {(clients_with_zero_ai / total_clients) * 100:.2f}%")

--- DATA LIMIT: CLIENT BIAS ---
Total Unique Clients: 32
Clients with exactly ZERO AI traffic across all pages: 6
Percentage of blind-spot clients: 18.75%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.